# Gross Disposable Household Income (GDHI) per head → ITL2

Builds **GDHI per head (£) per ITL2 region per year** and saves it to `clean/gdhi_itl2.csv`.

- **Source:** ONS *Regional gross disposable household income (GDHI)*, **Table 3 — GDHI per head (£)**.
- **What it is & why:** a **residence-based** measure of what households in an area actually have to spend (income after tax and benefits, per person). It complements the workplace-based **GVA** indicator: the two diverge where commuting matters (e.g. Inner London's output per worker is huge, but residents' disposable income is lower). GDHI per head is the better **living-standards / local-prosperity** signal for RQ4. *(Not in the original application variable list — added as a complementary prosperity measure.)*
- **Simple source:** already at ITL2 (2025) — the 46 codes match the spine, so **no crosswalk, no aggregation**. Covers all 46 regions incl. Scotland & NI. Years 1997–2024.

The sheet mixes ITL levels in one table with an `ITL` column labelling each row (UK / ITL1 / **ITL2** / ITL3), so we filter to `ITL2`. Put the workbook in `raw/gdhi/`; edit only `BASE`.

In [ ]:
import re
from pathlib import Path
import pandas as pd

# --- the ONE thing to edit: your project root ---
BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")

GDHI_DIR = BASE / "raw" / "gdhi"        # put the ONS GDHI workbook here
SPINE    = BASE / "clean" / "geography_spine.csv"
OUT      = BASE / "clean" / "gdhi_itl2.csv"
SHEET    = "Table 3"                    # GDHI per head (£)

books = sorted(GDHI_DIR.glob("*.xlsx"))
if not books:
    raise SystemExit(f"No .xlsx in {GDHI_DIR}. Put the ONS regional GDHI workbook there.")
GDHI_FILE = books[-1]
print("Using:", GDHI_FILE.name)

## 1. Extract GDHI per head for ITL2 regions

Read Table 3, keep the rows labelled `ITL2`, and reshape the year columns into tidy rows.

In [ ]:
df = pd.read_excel(GDHI_FILE, sheet_name=SHEET, skiprows=3)
df.columns = [str(c).strip() for c in df.columns]

level_col = df.columns[0]                      # the 'ITL' level label column
itl2 = df[df[level_col].astype(str).str.strip() == "ITL2"].copy()

year_cols = [c for c in itl2.columns if re.fullmatch(r"\d{4}", str(c))]
gdhi = (itl2.melt(id_vars=["ITL code", "Region name"], value_vars=year_cols,
                  var_name="year", value_name="gdhi_per_head_gbp")
            .rename(columns={"ITL code": "itl2_code", "Region name": "itl2_name"}))
gdhi["year"] = gdhi["year"].astype(int)
gdhi["gdhi_per_head_gbp"] = pd.to_numeric(gdhi["gdhi_per_head_gbp"], errors="coerce")
gdhi = gdhi.sort_values(["year", "itl2_code"]).reset_index(drop=True)

print(f"{len(gdhi)} rows | {gdhi.itl2_code.nunique()} regions | years {gdhi.year.min()}-{gdhi.year.max()}")
gdhi.head()

## 2. Check codes match the spine, then save

Confirm every ITL2 code exists in the spine (these are ITL codes, so no crosswalk needed — just a consistency check).

In [ ]:
spine = pd.read_csv(SPINE)
spine_codes = set(spine.itl2_code.unique())
gdhi_codes = set(gdhi.itl2_code.unique())
print(f"Spine regions: {len(spine_codes)} | GDHI regions: {len(gdhi_codes)}")
print("GDHI codes not in spine:", (gdhi_codes - spine_codes) or "none")
print("Spine codes not in GDHI:", (spine_codes - gdhi_codes) or "none")

OUT.parent.mkdir(parents=True, exist_ok=True)
gdhi.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}")

## 3. Coverage & sanity check

All 46 regions should be present. The latest-year ranking should put Inner London / the Greater South East on top and poorer regions at the bottom — and note it's a *different* ranking from GVA (residence vs workplace).

In [ ]:
latest = gdhi.year.max()
print(f"Coverage: {gdhi.itl2_code.nunique()} of {len(spine_codes)} regions x "
      f"{gdhi.year.nunique()} years ({gdhi.year.min()}-{latest})")
y = gdhi[gdhi.year == latest]
print(f"\n{latest} highest / lowest GDHI per head (£):")
print(pd.concat([y.nlargest(3, "gdhi_per_head_gbp"), y.nsmallest(3, "gdhi_per_head_gbp")])
        [["itl2_code","itl2_name","gdhi_per_head_gbp"]].to_string(index=False))